# Laboratório — SGD e learning rate

Implementaremos SGD em NumPy e separaremos três perguntas: **o gradiente está correto?**, **o passo é estável?** e **o protocolo de seleção é honesto?**

**Dependências:** Python ≥ 3.11, NumPy ≥ 1.26 e Matplotlib ≥ 3.8.  
**Seed-base:** `20260918`.  
**Dados:** regressão linear sintética com splits independentes; nenhum download ou segredo.


## 1. Ambiente

Avisos numéricos são promovidos a erros. O notebook comitado permanece sem outputs depois desta execução de validação.


In [ ]:
import platform
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("error")
BASE_SEED = 20260918
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")


## 2. Quadrática unidimensional: o limite exato

Para $J(\theta)=\tfrac12\lambda\theta^2$, a trajetória é $\theta_t=(1-\eta\lambda)^t\theta_0$. Com $\lambda=10$, o limite estrito é $\eta<0{,}2$.


In [ ]:
def quadratic_trajectory(theta0, curvature, learning_rate, steps):
    theta = float(theta0)
    trajectory = [theta]
    losses = [0.5 * curvature * theta**2]
    for _ in range(steps):
        grad = curvature * theta
        theta -= learning_rate * grad
        trajectory.append(theta)
        losses.append(0.5 * curvature * theta**2)
    return np.array(trajectory), np.array(losses)

curvature = 10.0
etas_1d = [0.02, 0.10, 0.15, 0.20, 0.21]
quad_runs = {eta: quadratic_trajectory(4.0, curvature, eta, 25) for eta in etas_1d}

assert np.all(np.diff(quad_runs[0.02][1]) < 0)
assert np.isclose(quad_runs[0.10][0][1], 0.0)
assert np.all(np.sign(quad_runs[0.15][0][1:6]) == np.array([-1, 1, -1, 1, -1]))
assert np.allclose(np.abs(quad_runs[0.20][0]), 4.0)
assert quad_runs[0.21][1][-1] > quad_runs[0.21][1][0]
for eta in etas_1d:
    theta, loss = quad_runs[eta]
    print(f"eta={eta:.2f}: multiplicador={1-eta*curvature:+.2f}; theta_25={theta[-1]:+.6f}; loss_25={loss[-1]:.6f}")


### Gráfico: cinco regimes de learning rate

**Texto alternativo:** escala logarítmica da loss por step; as curvas de 0,02, 0,10 e 0,15 descem, 0,20 permanece horizontal e 0,21 cresce.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for eta, (_, losses) in quad_runs.items():
    ax.semilogy(np.maximum(losses, 1e-18), label=f"eta={eta}")
ax.set(xlabel="step", ylabel="loss", title="Estabilidade em J(theta)=5 theta²")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
fig.tight_layout()
plt.show()
plt.close(fig)


## 3. Quadrática bidimensional mal condicionada

Com Hessiana diagonal $(1,100)$, o passo deve satisfazer $\eta<0{,}02$. A direção de curvatura 1 progride lentamente quando o learning rate é limitado pela direção íngreme.


In [ ]:
H = np.diag([1.0, 100.0])
eigvals = np.linalg.eigvalsh(H)
stability_limit_2d = 2.0 / eigvals.max()

def run_quadratic_2d(eta, steps=200):
    theta = np.array([4.0, 4.0])
    losses = []
    for _ in range(steps):
        losses.append(0.5 * theta @ H @ theta)
        theta = theta - eta * (H @ theta)
    return theta, np.array(losses)

theta_stable, loss_stable = run_quadratic_2d(0.015)
theta_unstable, loss_unstable = run_quadratic_2d(0.0205, steps=60)
assert np.isclose(stability_limit_2d, 0.02)
assert loss_stable[-1] < loss_stable[0]
assert loss_unstable[-1] > loss_unstable[0]
assert abs(theta_stable[0]) > abs(theta_stable[1])
print(f"Autovalores: {eigvals.tolist()}; kappa={eigvals.max()/eigvals.min():.0f}")
print(f"Limite estável estrito: eta < {stability_limit_2d:.5f}")
print("Após 200 steps estáveis, |theta|:", np.round(np.abs(theta_stable), 6))


## 4. Dados sintéticos e splits preservados

Treino, validação e teste são gerados separadamente. Média e desvio são ajustados somente no treino; o teste não será consultado durante a escolha de $\eta$.


In [ ]:
rng_data = np.random.default_rng(BASE_SEED)
D = 5
w_true = np.array([1.7, -2.2, 0.6, 1.1, -0.8])
b_true = 0.35
feature_scales = np.array([1.0, 4.0, 0.25, 2.0, 0.7])

def make_split(n, rng):
    X_raw = rng.normal(size=(n, D)) * feature_scales
    y = X_raw @ w_true + b_true + rng.normal(scale=0.45, size=n)
    return X_raw, y

X_train_raw, y_train = make_split(320, rng_data)
X_val_raw, y_val = make_split(120, rng_data)
X_test_raw, y_test = make_split(120, rng_data)

mean_train = X_train_raw.mean(axis=0)
std_train = X_train_raw.std(axis=0)
X_train = (X_train_raw - mean_train) / std_train
X_val = (X_val_raw - mean_train) / std_train
X_test = (X_test_raw - mean_train) / std_train

def add_bias(X):
    return np.column_stack([X, np.ones(len(X))])

X_train_b, X_val_b, X_test_b = map(add_bias, (X_train, X_val, X_test))
assert X_train_b.shape == (320, D + 1)
assert X_val_b.shape == X_test_b.shape == (120, D + 1)
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-14)
assert np.allclose(X_train.std(axis=0), 1.0, atol=1e-14)
print(f"Treino={X_train_b.shape}; validação={X_val_b.shape}; teste reservado={X_test_b.shape}")


## 5. Iterador reproduzível

Usamos a política da Aula 17: uma permutação derivada de `(seed, epoch)` e último lote preservado.


In [ ]:
def batch_indices(n, batch_size, *, seed, epoch):
    rng = np.random.default_rng(np.random.SeedSequence([seed, epoch]))
    order = rng.permutation(n)
    for start in range(0, n, batch_size):
        yield order[start:min(start + batch_size, n)]

batches_0 = list(batch_indices(len(X_train_b), 32, seed=BASE_SEED, epoch=0))
flat_0 = np.concatenate(batches_0)
assert len(batches_0) == 10
assert np.array_equal(np.sort(flat_0), np.arange(320))
assert len(np.unique(flat_0)) == 320
print(f"Epoch 0: {len(batches_0)} batches; cobertura={len(flat_0)}/320; duplicações=0")


## 6. Loss, gradiente e contrato do step

Usamos MSE com fator $1/2$: $J(\theta)=\tfrac{1}{2B}\|X\theta-y\|^2$. O gradiente médio é $X^\top(X\theta-y)/B$.


In [ ]:
def mse_half(X, y, theta):
    residual = X @ theta - y
    return 0.5 * np.mean(residual**2)

def mse_gradient(X, y, theta):
    return X.T @ (X @ theta - y) / len(X)

def sgd_step(theta, grad, learning_rate):
    if theta.shape != grad.shape:
        raise ValueError("theta e grad devem ter o mesmo shape")
    if learning_rate <= 0:
        raise ValueError("learning_rate deve ser positivo")
    if not np.isfinite(grad).all():
        raise FloatingPointError("gradiente não finito")
    return theta - learning_rate * grad

theta_probe = np.zeros(D + 1)
grad_probe = mse_gradient(X_train_b[:32], y_train[:32], theta_probe)
loss_before = mse_half(X_train_b[:32], y_train[:32], theta_probe)
theta_probe_1 = sgd_step(theta_probe, grad_probe, 1e-3)
loss_after = mse_half(X_train_b[:32], y_train[:32], theta_probe_1)
assert theta_probe_1.shape == theta_probe.shape
assert loss_after < loss_before
print(f"Step pequeno: loss {loss_before:.6f} -> {loss_after:.6f}; ||grad||={np.linalg.norm(grad_probe):.6f}")


## 7. Limite da quadrática de treino

Na regressão linear, a Hessiana é $X^\top X/N$. Isso permite calcular a faixa estável do gradient descent full-batch e testar ambos os lados do limite.


In [ ]:
hessian = X_train_b.T @ X_train_b / len(X_train_b)
lambda_max = np.linalg.eigvalsh(hessian).max()
eta_limit_train = 2.0 / lambda_max

def full_batch_descent(eta, steps=80):
    theta = np.zeros(D + 1)
    losses = [mse_half(X_train_b, y_train, theta)]
    for _ in range(steps):
        theta = sgd_step(theta, mse_gradient(X_train_b, y_train, theta), eta)
        losses.append(mse_half(X_train_b, y_train, theta))
        if losses[-1] > 1e12:
            break
    return theta, np.array(losses)

theta_safe, full_safe = full_batch_descent(0.90 * eta_limit_train)
theta_bad, full_bad = full_batch_descent(1.05 * eta_limit_train)
assert full_safe[-1] < 0.02 * full_safe[0]
assert full_bad[-1] > full_bad[0]
print(f"lambda_max={lambda_max:.6f}; limite full-batch={eta_limit_train:.6f}")
print(f"eta=0,90×limite: loss final={full_safe[-1]:.6f}")
print(f"eta=1,05×limite: loss final={full_bad[-1]:.3e} após {len(full_bad)-1} steps")


## 8. Laço de mini-batch SGD

O histórico separa loss do batch antes do update e loss completa no fim da epoch. A função interrompe execuções claramente divergentes antes de overflow.


In [ ]:
def train_sgd(learning_rate, *, epochs=80, batch_size=32, seed=BASE_SEED, schedule=None):
    theta = np.zeros(D + 1)
    batch_losses, epoch_train, epoch_val, update_ratios, lrs = [], [], [], [], []
    step = 0
    diverged = False
    for epoch in range(epochs):
        for idx in batch_indices(len(X_train_b), batch_size, seed=seed, epoch=epoch):
            eta_t = learning_rate if schedule is None else schedule(learning_rate, step)
            loss_b = mse_half(X_train_b[idx], y_train[idx], theta)
            grad = mse_gradient(X_train_b[idx], y_train[idx], theta)
            update = -eta_t * grad
            ratio = np.linalg.norm(update) / (np.linalg.norm(theta) + 1e-12)
            theta = sgd_step(theta, grad, eta_t)
            batch_losses.append(loss_b)
            update_ratios.append(ratio)
            lrs.append(eta_t)
            step += 1
            if not np.isfinite(theta).all() or np.linalg.norm(theta) > 1e6:
                diverged = True
                break
        if diverged:
            break
        train_loss = mse_half(X_train_b, y_train, theta)
        val_loss = mse_half(X_val_b, y_val, theta)
        epoch_train.append(train_loss)
        epoch_val.append(val_loss)
        if train_loss > 1e12:
            diverged = True
            break
    return {
        "theta": theta, "batch_loss": np.array(batch_losses),
        "train_loss": np.array(epoch_train), "val_loss": np.array(epoch_val),
        "update_ratio": np.array(update_ratios), "lr": np.array(lrs),
        "steps": step, "diverged": diverged,
    }

probe_run = train_sgd(0.05, epochs=3)
assert probe_run["steps"] == 30
assert len(probe_run["train_loss"]) == 3
assert not probe_run["diverged"]
print(f"Ensaio: {probe_run['steps']} steps; loss treino={probe_run['train_loss'][-1]:.6f}")


## 9. Busca logarítmica sem consultar o teste

Todos os candidatos usam o mesmo split, orçamento, inicialização e ordem de batches. A decisão usa somente a menor loss de validação.


In [ ]:
lr_candidates = np.array([1e-4, 1e-3, 1e-2, 5e-2, 2e-1, 8e-1, 2.0])
lr_runs = {}
validation_scores = []

for eta in lr_candidates:
    run = train_sgd(float(eta), epochs=80, batch_size=32, seed=BASE_SEED)
    lr_runs[float(eta)] = run
    score = np.inf if run["diverged"] or len(run["val_loss"]) == 0 else run["val_loss"][-1]
    validation_scores.append(score)
    label = "divergiu" if not np.isfinite(score) else f"val={score:.6f}"
    print(f"eta={eta:g}: steps={run['steps']}; {label}")

validation_scores = np.array(validation_scores)
best_index = int(np.argmin(validation_scores))
best_lr = float(lr_candidates[best_index])
best_run = lr_runs[best_lr]
assert np.isfinite(validation_scores[best_index])
assert best_lr not in (1e-4, 2.0)
assert lr_runs[2.0]["diverged"]
print(f"Selecionado na validação: eta={best_lr:g}; loss={validation_scores[best_index]:.6f}")


### Gráfico: curvas por epoch

**Texto alternativo:** learning rates muito pequenos descem lentamente, valores intermediários chegam à região de menor loss e o maior candidato diverge antes de completar o orçamento.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for eta in lr_candidates:
    run = lr_runs[float(eta)]
    if len(run["val_loss"]):
        ax.semilogy(np.arange(1, len(run["val_loss"]) + 1), run["val_loss"], label=f"eta={eta:g}")
ax.set(xlabel="epoch", ylabel="loss de validação", title="Seleção de learning rate")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)
fig.tight_layout()
plt.show()
plt.close(fig)


## 10. Teste reservado: uma consulta depois da decisão

Somente agora avaliamos o candidato selecionado. Também comparamos com a solução de mínimos quadrados como referência numérica, não como instrumento de seleção.


In [ ]:
test_loss_selected = mse_half(X_test_b, y_test, best_run["theta"])
theta_closed, *_ = np.linalg.lstsq(X_train_b, y_train, rcond=None)
test_loss_closed = mse_half(X_test_b, y_test, theta_closed)
parameter_gap = np.linalg.norm(best_run["theta"] - theta_closed)

assert test_loss_selected < 0.20
assert abs(test_loss_selected - test_loss_closed) < 0.03
assert parameter_gap < 0.15
print(f"Teste reservado SGD: {test_loss_selected:.6f}")
print(f"Teste mínimos quadrados: {test_loss_closed:.6f}")
print(f"Distância entre parâmetros: {parameter_gap:.6f}")


## 11. Schedule e noise floor

Em uma quadrática com ruído aditivo no gradiente, comparamos learning rate constante com $\eta_t=0{,}5(1+t)^{-0{,}6}$ usando exatamente a mesma sequência de ruído.


In [ ]:
rng_noise = np.random.default_rng(BASE_SEED + 1)
noise = rng_noise.normal(size=12000)

def noisy_scalar_path(schedule):
    theta = 4.0
    path, etas = [], []
    for t, epsilon in enumerate(noise):
        eta = schedule(t)
        theta -= eta * (theta + epsilon)
        path.append(theta)
        etas.append(eta)
    return np.array(path), np.array(etas)

constant_path, constant_etas = noisy_scalar_path(lambda t: 0.05)
decay_path, decay_etas = noisy_scalar_path(lambda t: 0.5 * (1 + t) ** -0.6)
constant_tail_rms = np.sqrt(np.mean(constant_path[-2000:] ** 2))
decay_tail_rms = np.sqrt(np.mean(decay_path[-2000:] ** 2))

assert decay_etas[-1] < decay_etas[0]
assert decay_tail_rms < 0.50 * constant_tail_rms
print(f"RMS final constante: {constant_tail_rms:.6f}")
print(f"RMS final decaimento: {decay_tail_rms:.6f}")
print(f"eta decaiu de {decay_etas[0]:.6f} para {decay_etas[-1]:.6f}")


## 12. Reprodutibilidade e efeito da seed do sampler

Mesma seed deve reproduzir bit a bit a trajetória. Outra seed muda a ordem dos mini-batches, mas uma configuração robusta continua chegando a uma região semelhante.


In [ ]:
run_same_a = train_sgd(best_lr, epochs=80, seed=BASE_SEED)
run_same_b = train_sgd(best_lr, epochs=80, seed=BASE_SEED)
run_other = train_sgd(best_lr, epochs=80, seed=BASE_SEED + 99)

assert np.array_equal(run_same_a["theta"], run_same_b["theta"])
assert np.array_equal(run_same_a["batch_loss"], run_same_b["batch_loss"])
assert not np.array_equal(run_same_a["theta"], run_other["theta"])
assert abs(run_same_a["val_loss"][-1] - run_other["val_loss"][-1]) < 0.02
print("Mesma seed, parâmetros idênticos:", np.array_equal(run_same_a["theta"], run_same_b["theta"]))
print(f"Outra seed: val={run_other['val_loss'][-1]:.6f}; referência={run_same_a['val_loss'][-1]:.6f}")


## 13. Epoch não fixa o número de updates entre batch sizes

Com 320 exemplos, uma epoch contém 40 updates para $B=8$ e 5 para $B=64$. Comparar apenas epochs confunde batch size com orçamento de atualização.


In [ ]:
updates_b8 = len(list(batch_indices(320, 8, seed=BASE_SEED, epoch=0)))
updates_b64 = len(list(batch_indices(320, 64, seed=BASE_SEED, epoch=0)))
assert updates_b8 == 40 and updates_b64 == 5
assert updates_b8 / updates_b64 == 8
print(f"Updates por epoch: B=8 -> {updates_b8}; B=64 -> {updates_b64}; razão={updates_b8/updates_b64:.0f}×")


## 14. Loss média dos batches não é a loss do parâmetro final

Durante uma epoch, cada batch é avaliado antes de uma atualização diferente. Por isso, a média desses valores não coincide com a avaliação completa no fim da epoch.


In [ ]:
one_epoch = train_sgd(best_lr, epochs=1, batch_size=32, seed=BASE_SEED)
mean_observed_batch_loss = one_epoch["batch_loss"].mean()
final_full_train_loss = one_epoch["train_loss"][-1]
assert not np.isclose(mean_observed_batch_loss, final_full_train_loss, rtol=1e-3)
assert final_full_train_loss < mean_observed_batch_loss
print(f"Média das losses pré-update: {mean_observed_batch_loss:.6f}")
print(f"Loss completa no parâmetro final: {final_full_train_loss:.6f}")


## 15. Auditoria final

Os contratos verificam teoria quadrática, shapes, splits, cobertura, estabilidade, seleção, teste reservado, schedule, seeds e semântica das curvas.


In [ ]:
checks = {
    "quadratica_monotona": np.all(np.diff(quad_runs[0.02][1]) < 0),
    "quadratica_um_step": np.isclose(quad_runs[0.10][0][1], 0.0),
    "quadratica_oscilatoria": abs(quad_runs[0.15][0][-1]) < 4.0,
    "fronteira_marginal": np.allclose(np.abs(quad_runs[0.20][0]), 4.0),
    "quadratica_diverge": quad_runs[0.21][1][-1] > quad_runs[0.21][1][0],
    "limite_2d": np.isclose(stability_limit_2d, 0.02),
    "direcao_plana_lenta": abs(theta_stable[0]) > abs(theta_stable[1]),
    "shape_treino": X_train_b.shape == (320, 6),
    "scaler_so_treino": np.allclose(X_train.mean(axis=0), 0.0, atol=1e-14),
    "cobertura": np.array_equal(np.sort(flat_0), np.arange(320)),
    "sem_duplicacao": len(np.unique(flat_0)) == 320,
    "step_reduz_loss": loss_after < loss_before,
    "limite_empirico_seguro": full_safe[-1] < 0.02 * full_safe[0],
    "acima_limite_diverge": full_bad[-1] > full_bad[0],
    "loop_30_steps": probe_run["steps"] == 30,
    "selecao_finita": np.isfinite(validation_scores[best_index]),
    "extremo_diverge": lr_runs[2.0]["diverged"],
    "teste_bom": test_loss_selected < 0.20,
    "sgd_proximo_fechada": parameter_gap < 0.15,
    "schedule_decresce": decay_etas[-1] < decay_etas[0],
    "schedule_reduz_noise_floor": decay_tail_rms < 0.50 * constant_tail_rms,
    "mesma_seed": np.array_equal(run_same_a["theta"], run_same_b["theta"]),
    "outra_seed": not np.array_equal(run_same_a["theta"], run_other["theta"]),
    "orcamento_batch": updates_b8 == 8 * updates_b64,
    "losses_semanticas_distintas": not np.isclose(mean_observed_batch_loss, final_full_train_loss, rtol=1e-3),
    "tudo_finito": np.isfinite(best_run["theta"]).all(),
}
failed = [name for name, ok in checks.items() if not ok]
assert not failed, failed
print(f"Auditoria: {sum(checks.values())}/{len(checks)} contratos aprovados.")


## Conclusão

O laboratório confirmou que:

- o limite quadrático separa convergência, oscilação marginal e divergência;
- a maior curvatura limita o learning rate global;
- a busca usa validação e consulta o teste somente após a decisão;
- o schedule decrescente reduz o noise floor no experimento controlado;
- seed do sampler, batch size, steps e semântica da loss fazem parte da evidência.

Na Aula 19, adicionaremos velocidade ao estado do otimizador com momentum e Nesterov.
